# Chapter 4 gallery, robust M regression

Reproduces `shock.R` (Ex 4.1) and `oats.R` (Ex 4.2). Robust fits via `lmrob_m`; robust nested-model testing via `rob_linear_test`.

In [ ]:
import os, sys, pathlib


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstattm_py as rpm
from robstattm_py import set_seed
from robstattm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## shock, robust M regression (Example 4.1)

`shock.R` fits LS, LS-without-outliers, L1 and a robust M-estimator (`lmrobM`) of average reaction `time` on `n.shocks`. Observations 1, 2, 4 are outliers that drag the LS line up.

In [ ]:
shock = rpm.datasets.shock()
print('columns:', list(shock.columns))
# robust M fit (bisquare, eff 0.85, bb 0.5 as in shock.R's control)
# The data frame is pushed to R with its original names (n.shocks), so
# the formula uses the R name; pandas column access uses n_shocks.
mfit = rpm.lmrob_m('time ~ n.shocks', shock, bb=0.5, efficiency=0.85, family='bisquare')
print('robust M coefficients:', np.round(mfit.coefficients, 4))
print('robust scale         :', round(float(mfit.scale), 4))

# LS on full data and on data without obs 1,2,4 (0-based: 0,1,3)
import numpy as np
Xls = np.c_[np.ones(len(shock)), shock['n_shocks'].to_numpy(float)]
yls = shock['time'].to_numpy(float)
b_full = np.linalg.lstsq(Xls, yls, rcond=None)[0]
keep = [i for i in range(len(shock)) if i not in (0, 1, 3)]
b_clean = np.linalg.lstsq(Xls[keep], yls[keep], rcond=None)[0]
print('LS (full)            :', np.round(b_full, 4))
print('LS (no obs 1,2,4)    :', np.round(b_clean, 4))

In [ ]:
# Figure 4.3 analogue: data + the three lines
xs = shock['n_shocks'].to_numpy(float); ys = shock['time'].to_numpy(float)
grid = np.linspace(xs.min(), xs.max(), 50)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(xs, ys, c='k', zorder=3)
ax.plot(grid, b_full[0] + b_full[1]*grid, 'r-', label='LS (full)')
ax.plot(grid, b_clean[0] + b_clean[1]*grid, color='gray', label='LS (no 1,2,4)')
ax.plot(grid, mfit.coefficients[0] + mfit.coefficients[1]*grid, 'g-', lw=2, label='robust M')
for i in (0, 1, 3):
    ax.annotate(str(i+1), (xs[i], ys[i]-0.4))
ax.set_xlabel('number of shocks'); ax.set_ylabel('average time'); ax.legend()
fig.savefig(FIG_DIR / 'ch4_shock.png', dpi=110, bbox_inches='tight'); plt.close(fig)
print('robust line resists the three outliers that pull LS up')

### Strict-tier cross-check vs direct R `lmrobM`

In [ ]:
# lmrobM is deterministic, so no seeding is needed on either side.
mfit_chk = rpm.lmrob_m('time ~ n.shocks', shock, bb=0.5, efficiency=0.85, family='bisquare')
ro.r('data(shock); cont <- lmrobM.control(bb=0.5, efficiency=0.85, family="bisquare")')
ro.r('rm_fit <- lmrobM(time ~ n.shocks, data=shock, control=cont)')
r_coef = np.asarray(ro.r('as.numeric(rm_fit$coefficients)'), dtype=float)
print('coefficients bit-equal to R:', np.array_equal(mfit_chk.coefficients, r_coef))

## oats, robust M regression + robust ANOVA (Example 4.2)

`oats.R` fits `lmrobM` models for two responses and compares nested models with `rob.linear.test` (robust analogue of the F-test). Our `rob_linear_test` wraps `rob.linear.test` for `lmrobdetMM` fits, so we demonstrate the robust nested-model test on MM fits of the oats data.

In [ ]:
oats = rpm.datasets.oats()
print('columns:', list(oats.columns))
# robust M fits (matching oats.R)
o2M = rpm.lmrob_m('response2 ~ variety + block', oats, bb=0.5, efficiency=0.85, family='bisquare')
print('response2 robust M scale:', round(float(o2M.scale), 4))
print('coefficients:', np.round(o2M.coefficients, 3))

In [ ]:
# Robust nested-model test (variety effect) via MM fits, which our
# rob_linear_test supports. Full = variety+block, reduced = block only.
full = rpm.lmrobdet_mm('response2 ~ variety + block', oats)
reduced = rpm.lmrobdet_mm('response2 ~ block', oats)
test = rpm.rob_linear_test(full, reduced)
print(test)
print(f'robust F p-value for the variety effect: {test.f_pvalue:.4f}')